In [2]:
import os
# List files in the current working directory to find the dataset
print(os.listdir('.'))

['.config', 'sample_data']


In [3]:
import os
if os.path.exists('sample_data'):
    print("Files in sample_data:", os.listdir('sample_data'))

Files in sample_data: ['anscombe.json', 'README.md', 'california_housing_train.csv', 'mnist_test.csv', 'mnist_train_small.csv', 'california_housing_test.csv']


In [4]:
import glob
print("Found CSV files:", glob.glob('**/*.csv', recursive=True))

Found CSV files: ['sample_data/california_housing_train.csv', 'sample_data/mnist_test.csv', 'sample_data/mnist_train_small.csv', 'sample_data/california_housing_test.csv']


In [5]:
import numpy as np
import pandas as pd

# Seed for reproducibility
np.random.seed(42)
n_rows = 1000

# Generate synthetic data mimicking CTR prediction task
client_ids = [f"client_{i:03d}" for i in np.random.randint(1, 50, size=n_rows)]
feature_1 = np.random.uniform(10, 100, size=n_rows)
feature_2 = np.random.normal(5, 2, size=n_rows)
# Synthetic CTR target between 0.0 and 0.5
ctr = 0.05 + 0.001 * feature_1 + 0.01 * feature_2 + np.random.normal(0, 0.02, size=n_rows)
ctr = np.clip(ctr, 0.0, 1.0)

synthetic_df = pd.DataFrame({
    'client_id': client_ids,
    'feature_1': feature_1,
    'feature_2': feature_2,
    'ctr': ctr
})

synthetic_df.to_csv('your_data.csv', index=False)
print("Synthetic 'your_data.csv' successfully created with shape:", synthetic_df.shape)

Synthetic 'your_data.csv' successfully created with shape: (1000, 4)


# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/arshiyanaaz411/flyrank-ml-internship/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [6]:
import numpy as np
import pandas as pd
from sklearn.model_selection import KFold, GroupKFold, GroupShuffleSplit
from sklearn.metrics import mean_absolute_error
from sklearn.dummy import DummyRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import HistGradientBoostingRegressor

# ---- EDIT THESE (same as Week 5) ----
DATA_PATH = "your_data.csv"   # anonymized sample
TARGET = "ctr"
GROUP = "client_id"
TIME_COL = None               # e.g. "date" if you have one, else keep None
# -------------------------------------

df = pd.read_csv(DATA_PATH).dropna(subset=[TARGET, GROUP]).reset_index(drop=True)
FEATURES = [c for c in df.select_dtypes("number").columns if c not in (TARGET, GROUP)]
X, y, g = df[FEATURES], df[TARGET], df[GROUP]
print("rows:", len(df), "| features:", len(FEATURES), "| groups:", g.nunique())

rows: 1000 | features: 2 | groups: 49


**Spirit of this section:** the paper was written for a broad audience and states its own standards. These are questions I would ask to practice the next level of rigor, not a grade.

### Finding 1: ___ (one sentence, in my own words)
- **Where does the label come from?** ___ (e.g. how is CTR measured, over what period, from which source?)
- **Does the validation design support the claim?** ___ (e.g. were the same sites/pages in both train and test? is the gain shown on unseen groups?)
- **What I would want to see next:** ___ (e.g. a grouped or time-aware split, or a confidence range)

### Finding 2: ___ (one sentence, in my own words)
- **Where does the label come from?** ___
- **Does the validation design support the claim?** ___
- **What I would want to see next:** ___

**What I appreciated:** ___ (one concrete thing the paper did well, such as disclosed limits or clear definitions)## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


**Before:** a random row split. Rows from the same client can land in both train and test, so the model can pick up client-specific patterns. This can make the score look better than it would be on new clients.

**After:** a grouped split (GroupKFold by client). Every test fold contains only clients the model has not seen. This matches the real question: predicting CTR for new clients.

Same model, same features, same metric (MAE, lower is better). Only the split changes.

**Observed (measured here, directional):** ___ (fill after running: how much did MAE change from before to after?)## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

In [ ]:
def cv_mae(model_fn, splitter, **split_kw):
    maes = []
    for tr, va in splitter.split(X, y, **split_kw):
        m = model_fn()
        m.fit(X.iloc[tr], y.iloc[tr])
        maes.append(mean_absolute_error(y.iloc[va], m.predict(X.iloc[va])))
    return np.mean(maes), np.std(maes)

hgb_fn = lambda: HistGradientBoostingRegressor(max_iter=300, learning_rate=0.05, random_state=42)
base_fn = lambda: DummyRegressor(strategy="median")   # replace with your Week-4 baseline

rows = []
for label, splitter, kw in [
    ("Before: random row split", KFold(5, shuffle=True, random_state=42), {}),
    ("After: grouped by client", GroupKFold(5), {"groups": g}),
]:
    for mname, fn in [("Baseline", base_fn), ("Gradient Boosting", hgb_fn)]:
        mean, std = cv_mae(fn, splitter, **kw)
        rows.append({"split": label, "model": mname, "MAE": mean, "std": std})

res = pd.DataFrame(rows).round(4)
display(res.pivot(index="model", columns="split", values="MAE"))

# Optional: time-aware check, only if a time column exists
if TIME_COL:
    d = df.sort_values(TIME_COL).reset_index(drop=True)
    cut = int(len(d) * 0.8)
    tr, te = d.iloc[:cut], d.iloc[cut:]
    m = hgb_fn().fit(tr[FEATURES], tr[TARGET])
    print("Time-aware holdout MAE:", round(mean_absolute_error(te[TARGET], m.predict(te[FEATURES])), 4))

**Question:** does any feature carry information that would not be available before the click, or that gives away the target or the client?

**Checks run below:**
1. Feature names that look related to the target (for example clicks, impressions, or anything derived from CTR).
2. Single-feature test: a shallow tree on one feature at a time under grouped CV. A feature that alone gets close to solving the target is a leakage suspect.
3. Within-client constancy: features that barely change inside a client can act as a client ID, which helps under a random split but not on new clients.
4. Shuffled-target sanity check: with the target shuffled, MAE should be about the baseline. If it is much lower, something is wrong in the pipeline.
5. Re-run without the suspects and compare MAE.

**Observed (measured, directional):** ___ (which features were flagged, and what happened to MAE when they were removed)
**Decision:** ___ (keep / drop, with a one-line reason each)## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

In [ ]:
# 1) Suspicious names
keywords = ["click", "impress", "ctr", "target", "label", "rate"]
name_flags = [c for c in FEATURES if any(k in c.lower() for k in keywords)]
print("Name flags:", name_flags)

# 2) Single-feature grouped CV (shallow tree)
base_mean, _ = cv_mae(base_fn, GroupKFold(5), groups=g)
single = {}
for f in FEATURES:
    maes = []
    for tr, va in GroupKFold(5).split(X, y, groups=g):
        t = DecisionTreeRegressor(max_depth=4, min_samples_leaf=50, random_state=42)
        t.fit(X.iloc[tr][[f]].fillna(X[f].median()), y.iloc[tr])
        maes.append(mean_absolute_error(y.iloc[va], t.predict(X.iloc[va][[f]].fillna(X[f].median()))))
    single[f] = np.mean(maes)
single = pd.Series(single).sort_values()
print("Baseline MAE:", round(base_mean, 4))
print((single / base_mean).head(10).round(3).rename("single-feature MAE / baseline MAE"))

# 3) Within-client constancy (share of variance that sits between clients)
between = {}
for f in FEATURES:
    tot = df[f].var()
    between[f] = (df.groupby(GROUP)[f].mean().var() / tot) if tot and tot > 0 else np.nan
between = pd.Series(between).sort_values(ascending=False)
print(between.head(10).round(3).rename("between-client variance share"))

# 4) Shuffled-target check
rng = np.random.RandomState(0)
y_shuf = pd.Series(rng.permutation(y.values), index=y.index)
maes = []
for tr, va in GroupKFold(5).split(X, y_shuf, groups=g):
    m = hgb_fn().fit(X.iloc[tr], y_shuf.iloc[tr])
    maes.append(mean_absolute_error(y_shuf.iloc[va], m.predict(X.iloc[va])))
print("Shuffled-target MAE:", round(np.mean(maes), 4), "| baseline MAE:", round(base_mean, 4))

# 5) Re-run without suspects (edit the list after looking at the outputs above)
SUSPECTS = list(set(name_flags))   # add or remove names here
if SUSPECTS:
    keep = [c for c in FEATURES if c not in SUSPECTS]
    maes = []
    for tr, va in GroupKFold(5).split(X, y, groups=g):
        m = hgb_fn().fit(X.iloc[tr][keep], y.iloc[tr])
        maes.append(mean_absolute_error(y.iloc[va], m.predict(X.iloc[va][keep])))
    print("MAE without suspects:", round(np.mean(maes), 4))

Claims from Week 5 that went further than the evidence, and the rewrite:

| Week-5 wording | Problem | Rewritten (evidence-matched) |
|---|---|---|
| "The model predicts CTR well" | Based on a split that may share clients between train and test | "Under a grouped split, measured MAE was ___ vs baseline ___ (directional)" |
| "Feature ___ drives CTR" | Permutation importance shows use by the model, not cause | "The model relied on ___ most (measured); this does not show it causes CTR" |
| "Model is X% better" | One split, no spread shown | "Observed ___% lower MAE across 5 grouped folds (std ___)" |
| ___ | ___ | ___ |

**Limits I would state publicly:** one dataset sample, one time period, new-client performance only checked on held-out clients from the same sample. Use as decision-support, not as a guarantee.## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.